
# 1. (1.75 points)

Given the following quarterly exposure information:

|Calendar Year and Quarter|Written Exposures| Earned Exposures |
|-------------------------|-----------------|------------------|
|2017 Q1|100| 5                |
|2017 Q2|450| 247.5            |
|2017 Q3|400| 427.5            |
|2017 Q4|100|52.5|
|2018 Q1|125|53.75|
|2018 Q2|550|528.75|
|2018 Q3|475|562.5|
|2018 Q4|30|59|

- The company started writing business on January 1, 2017.
- The company stops writing business on December 31, 2018.
- The quarterly earnings pattern was set by analyzing historical experience across the industry and is not uniform.
- All policies are annual.
- All policies are written on the first day of the quarter.
- There are no policy cancellations and no mid-term adjustments.

* a. (0.5 point) Calculate the 2017 policy year earned exposures as of March 31, 2018.
* b. (0.25 point) Calculate the in-force exposures as of May 31, 2018.
* c. (0.5 point) Calculate the calendar year 2018 unearned exposures.
* d. (0.5 point) Calculate the calendar year 2019 quarter 1 earned exposures.

## Solution

Source: Werner & Modlin, Chapter 4 (Exposures): written, earned, unearned and in-force exposures. The relevant definitions are:

- **Earned exposure** is the portion of written exposure for which coverage has already been provided.
- **Calendar year unearned** = CY written $-$ CY earned $+$ unearned at the beginning of the calendar year.
- **In-force exposure** counts policies with inception on or before the date and expiration after it.
- The even-earning assumption does not hold for seasonal lines, so the actuary specifies another earning pattern.

**Step 1: back out the earning pattern.** All policies are annual and written on the first day of a quarter, so every policy covers each of the
four calendar quarters exactly once. The "non-uniform quarterly earning pattern" is therefore a share $f_q$ of a policy's exposure earned in
calendar quarter $q$, with $\sum_q f_q = 1$. Calendar quarter earned exposure = (exposure written in that quarter and the previous three) $\times f_q$.

In [1]:
import math
import numpy as np
import pandas as pd

quarters = pd.period_range('2017Q1', periods=8, freq='Q')
written = pd.Series([100, 450, 400, 100, 125, 550, 475, 30], index=quarters, dtype=float)
earned = pd.Series([5, 247.5, 427.5, 52.5, 53.75, 528.75, 562.5, 59], index=quarters)

# Exposure covered by policies that can earn in each quarter (written this quarter or the previous 3)
in_force_base = written.rolling(4, min_periods=1).sum()
implied = earned / in_force_base

pattern = implied.groupby(implied.index.quarter).agg(['min', 'max'])
pattern

,min,max
1,0.05,0.05
2,0.45,0.45
3,0.45,0.45
4,0.05,0.05


The implied share is the same in each calendar quarter in both years, so the pattern is by calendar quarter:
Q1 5%, Q2 45%, Q3 45%, Q4 5% (sums to 100%).

In [2]:
f = implied.groupby(implied.index.quarter).first().round(10)
assert math.isclose(f.sum(), 1.0)
f

1    0.05
2    0.45
3    0.45
4    0.05
dtype: float64

**Step 2: earned exposure by policy cohort and calendar quarter.** A cohort written on the first day of quarter $w$ earns $W_w \times f_q$ in
each of calendar quarters $w, w+1, w+2, w+3$.

In [3]:
calendar = pd.period_range('2017Q1', periods=11, freq='Q')   # runs through the last quarter any policy earns (2019Q3)
earned_matrix = pd.DataFrame(0.0, index=quarters, columns=calendar)
for cohort in quarters:
    for k in range(4):
        q = cohort + k
        earned_matrix.loc[cohort, q] = written[cohort] * f[q.quarter]

# the matrix reproduces the given earned exposures
assert np.allclose(earned_matrix.sum().loc[quarters], earned)
earned_matrix

,2017Q1,2017Q2,2017Q3,2017Q4,2018Q1,2018Q2,2018Q3,2018Q4,2019Q1,2019Q2,2019Q3
2017Q1,5.0,45.0,45.0,5.0,0.00,0.00,0.00,0.00,0.00,0.00,0.0
2017Q2,0.0,202.5,202.5,22.5,22.50,0.00,0.00,0.00,0.00,0.00,0.0
2017Q3,0.0,0.0,180.0,20.0,20.00,180.00,0.00,0.00,0.00,0.00,0.0
2017Q4,0.0,0.0,0.0,5.0,5.00,45.00,45.00,0.00,0.00,0.00,0.0
2018Q1,0.0,0.0,0.0,0.0,6.25,56.25,56.25,6.25,0.00,0.00,0.0
2018Q2,0.0,0.0,0.0,0.0,0.00,247.50,247.50,27.50,27.50,0.00,0.0
2018Q3,0.0,0.0,0.0,0.0,0.00,0.00,213.75,23.75,23.75,213.75,0.0
2018Q4,0.0,0.0,0.0,0.0,0.00,0.00,0.00,1.50,1.50,13.50,13.5


### a.

Policy year 2017 earned as of March 31, 2018 = earned from the four 2017 cohorts through 2018 Q1.

In [4]:
cohorts_2017 = quarters[quarters.year == 2017]
pytd_2017 = earned_matrix.loc[cohorts_2017, :pd.Period('2018Q1')].sum().sum()
assert math.isclose(pytd_2017, 780)
pytd_2017

np.float64(780.0)

**a.** 780. That is 732.5 earned in 2017 plus 47.5 earned in 2018 Q1 from the Q2 to Q4 2017 policies (950 $\times$ 5%).

### b.

In-force on May 31, 2018: policies with inception on or before 5/31/2018 and expiration after it. Annual policies written on April 1, 2017 or earlier
have expired by then.

In [5]:
as_of = pd.Timestamp('2018-05-31')
start = quarters.start_time
expiry = start + pd.DateOffset(years=1)        # coverage ends at the end of the day before this date
in_force = written[(start <= as_of) & (expiry > as_of)]
assert math.isclose(in_force.sum(), 1175)
in_force

2017Q3    400.0
2017Q4    100.0
2018Q1    125.0
2018Q2    550.0
Freq: Q-DEC, dtype: float64

**b.** 1,175 (the Q3 2017 through Q2 2018 cohorts: 400 + 100 + 125 + 550).

### c.

By the text's formula: CY unearned = CY written $-$ CY earned $+$ unearned at the beginning of the year.

In [6]:
cy_written = written.groupby(written.index.year).sum()
cy_earned = earned.groupby(earned.index.year).sum()

unearned_end_2017 = cy_written[2017] - cy_earned[2017]          # business started 1/1/2017, so nothing at the start
unearned_end_2018 = cy_written[2018] - cy_earned[2018] + unearned_end_2017

# cross-check from the cohort matrix: exposure written but not yet earned by 12/31/2018
check = written.sum() - earned_matrix.loc[:, :pd.Period('2018Q4')].sum().sum()
assert math.isclose(unearned_end_2018, 293.5) and math.isclose(check, 293.5)
pd.Series({'unearned 12/31/2017': unearned_end_2017, 'unearned 12/31/2018': unearned_end_2018})

unearned 12/31/2017    317.5
unearned 12/31/2018    293.5
dtype: float64

**c.** 293.5 (317.5 at the start of 2018, plus 1,180 written, minus 1,204 earned).

### d.

Calendar year 2019 Q1 earned = exposure from policies still in force (written in 2018 Q2 to Q4; the 2018 Q1 policies expired on 12/31/2018) $\times$ 5%.

In [7]:
q1_2019 = earned_matrix[pd.Period('2019Q1')].sum()
assert math.isclose(q1_2019, 52.75)
q1_2019

np.float64(52.75)

**d.** 52.75 (= (550 + 475 + 30) $\times$ 5%).

The remaining 293.5 of unearned exposure earns out as 52.75 in 2019 Q1, 227.25 in Q2 and 13.5 in Q3, which agrees with part c.

## chainladder-python gaps

**What `chainladder` covers.** Nothing for this problem. It works on loss triangles (origin $\times$ development), and everything here was done with pandas.

**What is missing.**

- No exposure or premium earning: nothing converts written exposure/premium into earned, unearned, or in-force amounts using a policy term and an earning pattern
  (even or otherwise). The calendar-quarter pattern had to be backed out and applied by hand.
- No policy-year vs calendar-year aggregation of the same underlying written exposure, which is what parts a and c turn on.
- No in-force snapshot at an arbitrary date.

**Suggested feature (probably out of scope for a reserving package, but a natural input to Cape Cod, Bornhuetter-Ferguson and Clark's method, which all need earned exposure/premium by origin).**
An earning helper that takes written exposure or premium by period, a policy term, and an optional earning pattern, and returns earned premium/exposure as a `Triangle`
by calendar period, policy period and valuation date. For example `cl.earn(written, term=12, pattern=None)`, with `.unearned` and `.in_force(date)` accessors. This would feed
the `sample_weight` argument of `ClarkLDF`, `CapeCod` and `BornhuetterFerguson` directly.